In [1]:
# %load_ext autoreload
# %autoreload 2
import os

import matplotlib.pyplot as plt
import torch
import torchdiffeq
import torchsde
from torchdyn.core import NeuralODE
from torchvision import datasets, transforms
from torchvision.transforms import ToPILImage
from torchvision.utils import make_grid
from tqdm import tqdm

sys.append("../..")


import sys
# sys.path.append("/mnt/homes/yijin/3D-Diffusion-Policy/3D-Diffusion-Policy/conditional_flow_matching/conditional-flow-matching")   # 这里需要将torchcfm 的路径添加到环境变量中，这样就可以使用本地的torchcfm, 而不是pip上面的

# from conditional_flow_matching.torchcfm.conditional_flow_matching import *
# from conditional_flow_matching.torchcfm.models.unet import UNetModel

from torchcfm.conditional_flow_matching import *
from torchcfm.models.unet import UNetModel

savedir = "models/cond_mnist"
os.makedirs(savedir, exist_ok=True)

ModuleNotFoundError: No module named 'numpy'

In [6]:
use_cuda = torch.cuda.is_available()
device = torch.device("cuda:3" if use_cuda else "cpu")
batch_size = 128
n_epochs = 10

trainset = datasets.MNIST(
    "../data",
    train=True,
    download=True,
    transform=transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))]),
)

train_loader = torch.utils.data.DataLoader(
    trainset, batch_size=batch_size, shuffle=True, drop_last=True
)

In [7]:
#################################
#    Class Conditional CFM
#################################

sigma = 0.0
model = UNetModel(
    dim=(1, 28, 28), num_channels=32, num_res_blocks=1, num_classes=10, class_cond=True
).to(device)

'''
看这里定义了 num_classes=10, 所以那个label_emb 是一个 num_classes = 10 的一个embed
'''

optimizer = torch.optim.Adam(model.parameters())
FM = ConditionalFlowMatcher(sigma=sigma)
# Users can try target FM by changing the above line by
# FM = TargetConditionalFlowMatcher(sigma=sigma)
node = NeuralODE(model, solver="dopri5", sensitivity="adjoint", atol=1e-4, rtol=1e-4)

In [8]:
for epoch in range(n_epochs):
    for i, data in enumerate(train_loader):
        optimizer.zero_grad()
        # 这里的x1 和 y 也是带有batch的
        # y 应该是label吧...
        x1 = data[0].to(device)
        y = data[1].to(device)
        # print(f"y[:10]: {y[:10]}")
        x0 = torch.randn_like(x1)
        t, xt, ut = FM.sample_location_and_conditional_flow(x0, x1)
        vt = model(t, xt, y)
        loss = torch.mean((vt - ut) ** 2)
        loss.backward()
        optimizer.step()
        print(f"epoch: {epoch}, steps: {i}, loss: {loss.item():.4}", end="\r")

# save models

torch.save(model.state_dict(), f"{savedir}/model.pth")
    

In [9]:

'''def forward(self, t, x, y=None):
        """Apply the model to an input batch.

        :param x: an [N x C x ...] Tensor of inputs.
        :param timesteps: a 1-D batch of timesteps.
        :param y: an [N] Tensor of labels, if class-conditional.
        :return: an [N x C x ...] Tensor of outputs.

        对嘛这里输入的是 100(batchsize) * 1(channel) * 28(H) * 28(W) 的噪声图像

        """'''


# load models
model.load_state_dict(torch.load(f"{savedir}/model.pth"))
model.eval()

USE_TORCH_DIFFEQ = True
generated_class_list = torch.arange(10, device=device).repeat(10)
with torch.no_grad():
    if USE_TORCH_DIFFEQ:
        traj = torchdiffeq.odeint(
            lambda t, x: model.forward(t, x, generated_class_list),
            torch.randn(100, 1, 28, 28, device=device),
            torch.linspace(0, 1, 2, device=device),
            atol=1e-4,
            rtol=1e-4,
            method="dopri5",
        )
    else:
        traj = node.trajectory(
            torch.randn(100, 1, 28, 28, device=device), # 这里应该是在采噪声吧
            t_span=torch.linspace(0, 1, 2, device=device),
        )
# print(f"torch.linspace(0, 1, 2, device=device): {torch.linspace(0, 1, 2, device=device)}") # tensor([0., 1.], device='cuda:3')

'''
详见unet forward函数我写的注释
这里会针对不同的t调用多次，我感觉应该是把那个unet交给odeint, 由 odeint 来调用的结果

'''




In [10]:


        

# print(f"generated_class_list[:20]: {generated_class_list[:20]}")
'''
generated_class_list[:20]: tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
       device='cuda:3')
'''
# print(f"traj.shape: {traj.shape}")
# traj[-1] 指的是生成的最后一个时间点的图像
# em但100个嘛，应该是说总的图像是100个，然后把它变成10*10的网格

'''
traj.shape: torch.Size([2, 100, 1, 28, 28])   2 应该对应的是时间点0和1， 100是图的数量，它这里应该是把100当成batchsize 相当于100个batch的东西搞进去这种感觉
'''

# noise_pic = torch.randn(100, 1, 28, 28, device=device)  
noise_pic = traj[0, :100]
''' 确实是noise pic'''

grid = make_grid(
    noise_pic.view([-1, 1, 28, 28]).clip(-1, 1), value_range=(-1, 1), padding=0, nrow=10
)
img = ToPILImage()(grid)
plt.imshow(img)
plt.show()
# plt.cla()

In [11]:

# visualize the generated numbers
grid = make_grid(
    traj[-1, :100].view([-1, 1, 28, 28]).clip(-1, 1), value_range=(-1, 1), padding=0, nrow=10
)
img = ToPILImage()(grid)
plt.imshow(img)
plt.show()

In [12]:
# ########################
# #   Use node to generate
# ########################

# 这里就不行了，因为node.trajectory 里面没有传入condition进去，所以在unet的条件生成的时候会报错

# traj = node.trajectory(
#             torch.randn(100, 1, 28, 28, device=device), # 这里应该是在采噪声吧
#             t_span=torch.linspace(0, 1, 2, device=device),
#         )
# grid = make_grid(
#     traj[-1, :100].view([-1, 1, 28, 28]).clip(-1, 1), value_range=(-1, 1), padding=0, nrow=10
# )
# img = ToPILImage()(grid)
# plt.imshow(img)
# plt.show()

In [13]:
#################################
#            OT-CFM
#################################

sigma = 0.0
model = UNetModel(
    dim=(1, 28, 28), num_channels=32, num_res_blocks=1, num_classes=10, class_cond=True
).to(device)
optimizer = torch.optim.Adam(model.parameters())
FM = ExactOptimalTransportConditionalFlowMatcher(sigma=sigma)
node = NeuralODE(model, solver="dopri5", sensitivity="adjoint", atol=1e-4, rtol=1e-4)

In [14]:
for epoch in range(n_epochs):
    for i, data in enumerate(train_loader):
        optimizer.zero_grad()
        x1 = data[0].to(device)
        y = data[1].to(device)
        x0 = torch.randn_like(x1)
        t, xt, ut, _, y1 = FM.guided_sample_location_and_conditional_flow(x0, x1, y1=y)
        vt = model(t, xt, y1)
        loss = torch.mean((vt - ut) ** 2)
        loss.backward()
        optimizer.step()
        print(f"epoch: {epoch}, steps: {i}, loss: {loss.item():.4}", end="\r")

(i, j): (array([ 56,  37,  88,  50,  74, 101, 112, 126,  14,  29, 116,  56,  36,
        64,  98, 124,  55,  35,  11,  33, 121,  80,  19, 103,  97, 117,
        60,  18,   3,  52,  68,  27,  31,  21,  64,  97,  71,  88,  70,
       124, 110,  32,  73, 102, 124, 124, 110,  12,  23,  14,  80,  36,
        95,   1,  82,  38,  92,  84,  46,   6,  93,  44,  72,  83,  91,
       101,  29,  25,  39,  66,  11,  15,  58,  48, 116, 117,  18,  40,
       120,  76,  83,  49,  76,  75,  80, 122,  50, 111,  59,  87,  38,
       122,  13,  42,  74,  56,   6,  90,  10,  93,  65, 107, 120,  68,
        91,  76, 113,  71,  49,  75,  69, 111,  85,  25,  62,  57,  47,
        36, 116,  50,  78,   8, 125,  78, 113,  20,  86,  53]), array([125, 124,  88,  65, 116, 113, 123, 106, 100,  23,  96, 125,  97,
        87, 102,  22,  21, 126,  28,  49,  62,   2,  44,  51,  10,  35,
        72,  16, 108,  59,  11, 103,  69, 105,  87,  10,  95,  88, 127,
        22,  29,  60,  61,  89,  22,  22,  29,  27, 115, 100,  

KeyboardInterrupt: 

In [ ]:
USE_TORCH_DIFFEQ = True
generated_class_list = torch.arange(10, device=device).repeat(10)
with torch.no_grad():
    if USE_TORCH_DIFFEQ:
        traj = torchdiffeq.odeint(
            lambda t, x: model.forward(t, x, generated_class_list),
            torch.randn(100, 1, 28, 28, device=device),
            torch.linspace(0, 1, 2, device=device),
            atol=1e-4,
            rtol=1e-4,
            method="dopri5",
        )
    else:
        traj = node.trajectory(
            torch.randn(100, 1, 28, 28, device=device),
            t_span=torch.linspace(0, 1, 2, device=device),
        )
grid = make_grid(
    traj[-1, :100].view([-1, 1, 28, 28]).clip(-1, 1), value_range=(-1, 1), padding=0, nrow=10
)
img = ToPILImage()(grid)
plt.imshow(img)
plt.show()

In [ ]:
# #################################
# #            SF2M
# #################################
# batch_size = 128
# n_epochs = 10
# sigma = 0.1


# model = UNetModel(
#     dim=(1, 28, 28), num_channels=32, num_res_blocks=1, num_classes=10, class_cond=True
# ).to(device)
# score_model = UNetModel(
#     dim=(1, 28, 28), num_channels=32, num_res_blocks=1, num_classes=10, class_cond=True
# ).to(device)

# optimizer = torch.optim.Adam(list(model.parameters()) + list(score_model.parameters()))
# FM = SchrodingerBridgeConditionalFlowMatcher(sigma=sigma)
# node = NeuralODE(model, solver="dopri5", sensitivity="adjoint", atol=1e-4, rtol=1e-4)

In [ ]:
# for epoch in range(n_epochs):
#     for i, data in tqdm(enumerate(train_loader)):
#         optimizer.zero_grad()
#         x1 = data[0].to(device)
#         y = data[1].to(device)
#         x0 = torch.randn_like(x1)
#         t, xt, ut, _, y1, eps = FM.guided_sample_location_and_conditional_flow(
#             x0, x1, y1=y, return_noise=True
#         )
#         lambda_t = FM.compute_lambda(t)
#         vt = model(t, xt, y1)
#         st = score_model(t, xt, y1)
#         flow_loss = torch.mean((vt - ut) ** 2)
#         score_loss = torch.mean((lambda_t[:, None, None, None] * st + eps) ** 2)
#         loss = flow_loss + score_loss
#         loss.backward()
#         optimizer.step()

In [ ]:
# USE_TORCH_DIFFEQ = True
# generated_class_list = torch.arange(10, device=device).repeat(10)

# node = NeuralODE(model, solver="euler", sensitivity="adjoint", atol=1e-4, rtol=1e-4)
# # Evaluate the ODE
# with torch.no_grad():
#     if USE_TORCH_DIFFEQ:
#         traj = torchdiffeq.odeint(
#             lambda t, x: model.forward(t, x, generated_class_list),
#             torch.randn(100, 1, 28, 28, device=device),
#             torch.linspace(0, 1, 2, device=device),
#             atol=1e-4,
#             rtol=1e-4,
#             method="dopri5",
#         )
#     else:
#         traj = node.trajectory(
#             torch.randn(100, 1, 28, 28, device=device),
#             t_span=torch.linspace(0, 1, 2, device=device),
#         )
# grid = make_grid(
#     traj[-1, :100].view([-1, 1, 28, 28]).clip(-1, 1), value_range=(-1, 1), padding=0, nrow=10
# )
# img = ToPILImage()(grid)
# plt.imshow(img)

In [ ]:
# # follows example from https://github.com/google-research/torchsde/blob/master/examples/cont_ddpm.py


# class SDE(torch.nn.Module):
#     noise_type = "diagonal"
#     sde_type = "ito"

#     def __init__(self, ode_drift, score, labels=None, reverse=False, sigma=0.1):
#         super().__init__()
#         self.drift = ode_drift
#         self.score = score
#         self.reverse = reverse
#         self.labels = labels
#         self.sigma = sigma

#     # Drift

#     def f(self, t, y):
#         y = y.view(-1, 1, 28, 28)
#         if self.reverse:
#             t = 1 - t
#             return -self.drift(t, y, self.labels) + self.score(t, y, self.labels)
#         return self.drift(t, y, self.labels).flatten(start_dim=1) + self.score(
#             t, y, self.labels
#         ).flatten(start_dim=1)

#     # Diffusion
#     def g(self, t, y):
#         return torch.ones_like(y) * self.sigma

In [ ]:
# sde = SDE(model, score_model, labels=torch.arange(10, device=device).repeat(10), sigma=0.1)
# with torch.no_grad():
#     sde_traj = torchsde.sdeint(
#         sde,
#         # x0.view(x0.size(0), -1),
#         torch.randn(100, 1 * 28 * 28, device=device),
#         ts=torch.linspace(0, 1, 2, device=device),
#         dt=0.01,
#     )

In [ ]:
# grid = make_grid(
#     sde_traj[-1, :100].view([-1, 1, 28, 28]).clip(-1, 1), value_range=(-1, 1), padding=0, nrow=10
# )
# img = ToPILImage()(grid)
# plt.imshow(img)